# 05 · Linked lists

After this notebook you can build and draw a linked list, rewire pointers without losing nodes, and solve the classic interview problems — reverse (iterative and recursive), detect a cycle and find where it starts, middle node, merge two sorted lists, remove the n-th node from the end, palindrome check, intersection of two lists — and say when a linked list actually matters in Python (rarely: `deque`, LRU caches).

**Time:** ~35 min · **Runs:** offline on CPU in < 1 min · **Needs:** [04 · Stacks and queues](04_stacks_and_queues.ipynb) · **Theory:** [DSA basics course](../../dsa_basics/index.html), chapter 4 "Nodes & linked lists"

## Why this matters in interviews

- Linked-list questions test pointer discipline: can you rewire `next` references in the right order without dropping the rest of the list?
- Three reusable tricks cover almost all of them: a **dummy head**, **fast and slow pointers**, and **in-place reversal**.
- The design follow-up — "build an LRU cache" — is a hash map plus a doubly linked list.

## What interviewers ask

- "Reverse a linked list — iteratively, then recursively. What is the space cost of each?" (LeetCode 206)
- "Does this list have a cycle? Where does it start? Why does Floyd's algorithm work?" (LC 141, 142)
- "Return the middle node." (LC 876)
- "Merge two sorted lists." (LC 21)
- "Remove the n-th node from the end in one pass." (LC 19)
- "Is the list a palindrome, in O(1) extra space?" (LC 234)
- "Where do two lists intersect?" (LC 160)
- "Design an LRU cache with O(1) get and put." (LC 146)
- Follow-ups: "Why a dummy node?" · "Compare nodes by value or identity?" · "When would you use a linked list in Python at all?"

In [ ]:
import random
import sys
import time
from collections import OrderedDict, deque

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

random.seed(0)
rng = random.Random(1)
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=COLORS), "axes.grid": True, "grid.color": "#e1e0d9",
                     "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2})

## The pattern: nodes and pointers

**In plain English:** a linked list is a chain of nodes; each node holds a value and a reference to the next node. There is no index — to reach the k-th node you walk k steps (O(k)) — but once you *hold* a node, inserting or deleting after it is O(1): you only rewire one or two references.

```
 head
  |
  v
+---+---+    +---+---+    +---+---+
| 1 | *-+--->| 2 | *-+--->| 3 | / |      "/" = None: the end of the list
+---+---+    +---+---+    +---+---+
 val next
```

**Recognise the tricks:**

| Situation | Trick |
|---|---|
| the head itself might change (delete the first node, merge, insert in front) | **dummy head** node before the real head |
| middle, cycle, "n-th from the end", "k-th from the end" | **fast & slow pointers** (fast moves 2 steps, or starts n steps ahead) |
| reverse all or part, palindrome, reorder | **in-place reversal** with `prev`, `cur`, `nxt` |
| two lists meet somewhere | walk both, **switch heads** at the end |

**Templates:**

```python
dummy = ListNode(0, head)          # a node before head: no special case for "the first node"
...
return dummy.next

slow = fast = head
while fast and fast.next:          # when fast falls off the end, slow is at the middle
    slow, fast = slow.next, fast.next.next

prev, cur = None, head             # reversal: save next, point back, advance
while cur:
    nxt = cur.next; cur.next = prev; prev, cur = cur, nxt
```

The node class and three helpers — build from a Python list, convert back (with a guard against cycles), and draw:

In [ ]:
class ListNode:
    __slots__ = ("val", "next")

    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

    def __repr__(self):
        return f"ListNode({self.val})"


def from_list(values):
    dummy = tail = ListNode()
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next


def to_list(head, limit=100_000):
    out = []
    while head is not None:
        out.append(head.val)
        head = head.next
        if len(out) > limit:
            raise ValueError("list is longer than the limit - is there a cycle?")
    return out


def draw_lists(rows, title=None):
    """rows: list of (head, label, marks); marks maps a node to a pointer name drawn above it. Cycles get a red arc."""
    fig, axes = plt.subplots(len(rows), 1, figsize=(10, 1.15 * len(rows) + 0.4), squeeze=False)
    for ax, (head, label, marks) in zip(axes[:, 0], rows):
        nodes, index, node = [], {}, head
        while node is not None and id(node) not in index:
            index[id(node)] = len(nodes)
            nodes.append(node)
            node = node.next
        loop_to = index.get(id(node)) if node is not None else None
        for i, nd in enumerate(nodes):
            x = i * 1.6
            ax.add_patch(FancyBboxPatch((x, 0), 0.9, 0.6, boxstyle="round,pad=0.04", fc="#e7f0ff", ec="#3b5bdb"))
            ax.text(x + 0.45, 0.3, str(nd.val), ha="center", va="center", fontsize=11)
            if i < len(nodes) - 1:
                ax.annotate("", xy=(x + 1.56, 0.3), xytext=(x + 0.96, 0.3), arrowprops=dict(arrowstyle="->", lw=1.4))
            if marks and nd in marks:
                ax.text(x + 0.45, 0.82, marks[nd], ha="center", fontsize=9, color=COLORS[1], fontweight="bold")
        last_x = (len(nodes) - 1) * 1.6
        if not nodes:
            ax.text(0, 0.3, "None (empty list)", va="center", fontsize=10, color="#52514e")
        elif loop_to is None:
            ax.annotate("None", xy=(last_x + 1.25, 0.3), xytext=(last_x + 0.96, 0.3), va="center", fontsize=9,
                        color="#52514e", arrowprops=dict(arrowstyle="-", lw=0))
        else:
            ax.annotate("", xy=(loop_to * 1.6 + 0.45, -0.05), xytext=(last_x + 0.45, -0.05),
                        arrowprops=dict(arrowstyle="->", lw=1.6, color=COLORS[7], connectionstyle="arc3,rad=-0.3"))
        ax.text(-0.3, 0.3, label, ha="right", va="center", fontsize=10)
        ax.set_xlim(-3.4, max(len(nodes), 1) * 1.6 + 1)
        ax.set_ylim(-1.0, 1.2)
        ax.axis("off")
    if title:
        fig.suptitle(title)
    plt.show()


head = from_list([1, 2, 3, 4])
print("round trip:", to_list(head))
assert to_list(from_list([])) == [] and to_list(from_list([7])) == [7]
draw_lists([(head, "head", {head: "head"})], "A singly linked list: every node points to the next")

## Problem 1 · Reverse a linked list (LC 206)

**Statement:** reverse the list and return the new head. `1→2→3→4→5` → `5→4→3→2→1`.

**Brute force:** copy the values into a Python list, reverse, rebuild — O(n) extra space. **Optimal (iterative):** walk once with three references. The order of the four assignments is the whole problem: *save* `nxt` before you overwrite `cur.next`, or the rest of the list is lost.

**Recursive version:** reverse the rest, then hang the current node at the end of the reversed rest. Elegant, but it uses one stack frame per node: O(n) space, and Python stops at ~1000 frames.

In [ ]:
def reverse_iterative(head):
    prev, cur = None, head
    while cur:
        nxt = cur.next        # 1. save the rest
        cur.next = prev       # 2. point this node backwards
        prev = cur            # 3. advance prev ...
        cur = nxt             # 4. ... and cur
    return prev               # the old tail is the new head


def reverse_recursive(head):
    if head is None or head.next is None:
        return head
    new_head = reverse_recursive(head.next)   # reverse everything after head
    head.next.next = head                     # the node after head now points back to head
    head.next = None
    return new_head


for values in [[], [1], [1, 2], [1, 2, 3, 4, 5]]:
    assert to_list(reverse_iterative(from_list(values))) == values[::-1]
    assert to_list(reverse_recursive(from_list(values))) == values[::-1]

long_list = from_list(range(5_000))
assert to_list(reverse_iterative(long_list))[:3] == [4999, 4998, 4997]
try:
    reverse_recursive(from_list(range(5_000)))
    deep_ok = True
except RecursionError:
    deep_ok = False
print("iterative: fine on 5,000 nodes | recursive on 5,000 nodes:", "ok" if deep_ok else "RecursionError (limit " + str(sys.getrecursionlimit()) + ")")
assert not deep_ok

before = from_list([1, 2, 3, 4, 5])
after = reverse_iterative(from_list([1, 2, 3, 4, 5]))
draw_lists([(before, "before", {before: "head"}), (after, "after", {after: "new head"})], "Reversal rewires every next pointer")

**Complexity:** iterative O(n) time, O(1) space; recursive O(n) time, **O(n) stack space**. In Python, say you would ship the iterative one.

## Problem 2 · Detect a cycle and find where it starts (LC 141, 142)

**Statement:** does following `next` ever loop? If so, return the node where the loop begins.

**Brute force:** remember every node you visit in a set (by identity) — the first repeat is the cycle start. O(n) time, O(n) space. **Optimal — Floyd's tortoise and hare:** `slow` moves 1 step, `fast` moves 2. With no cycle, `fast` falls off the end. With a cycle, `fast` gains one step per move inside the loop, so it must land on `slow`.

**Why resetting one pointer finds the start:** let `a` = steps from head to the cycle start, `b` = steps from the start to the meeting point, `c` = cycle length. At the meeting point slow has walked `a + b`, fast `2(a + b)`, and fast's extra distance is whole laps: `a + b = k·c`. So `a = k·c − b`: walking `a` more steps from the meeting point lands exactly on the cycle start — the same `a` steps a pointer from the head needs. Move one pointer from the head and one from the meeting point, one step each; they meet at the start.

In [ ]:
def make_cycle_list(values, pos):
    """Linked list whose tail points back to the node at index pos (pos = -1: no cycle)."""
    head = from_list(values)
    if pos < 0 or head is None:
        return head
    nodes, node = [], head
    while node:
        nodes.append(node)
        node = node.next
    nodes[-1].next = nodes[pos]
    return head


def cycle_start_brute(head):
    seen = set()
    while head is not None:
        if id(head) in seen:
            return head
        seen.add(id(head))
        head = head.next
    return None


def cycle_start(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:                     # they met inside the cycle
            finder = head
            while finder is not slow:        # a steps from the head == a steps from the meeting point
                finder, slow = finder.next, slow.next
            return finder
    return None


def has_cycle(head):
    return cycle_start(head) is not None


for values, pos in [([3, 2, 0, -4], 1), ([1, 2], 0), ([1], 0), ([1], -1), ([], -1), ([1, 2, 3, 4, 5, 6], 3)]:
    head = make_cycle_list(values, pos)
    start = cycle_start(head)
    assert start is cycle_start_brute(head)
    assert (start.val if start else None) == (values[pos] if pos >= 0 else None)
    assert has_cycle(head) == (pos >= 0)
for _ in range(300):
    n = rng.randint(1, 15)
    head = make_cycle_list(list(range(n)), rng.randint(-1, n - 1))
    assert cycle_start(head) is cycle_start_brute(head)
print("cycle_start (Floyd, O(1) space) matches the set-of-visited brute force on all cases")

In [ ]:
head = make_cycle_list([1, 2, 3, 4, 5, 6], 2)
slow = fast = head
while True:
    slow, fast = slow.next, fast.next.next
    if slow is fast:
        break
draw_lists([(head, "head", {head: "head", cycle_start(head): "cycle start", slow: "slow = fast meet"})],
           "Floyd's cycle detection: where slow and fast meet, and where the cycle starts")

**Complexity:** O(n) time, O(1) space. **Traps:** compare nodes with `is` (identity), never by value — two different nodes can hold the same value; check `fast and fast.next` before stepping twice.

## Problem 3 · Middle of the list (LC 876)

**Statement:** return the middle node; for an even length, the **second** of the two middles. `1→2→3→4→5` → 3; `1→2→3→4→5→6` → 4.

**Brute force:** count the nodes, then walk n // 2 steps — two passes. **Optimal:** fast and slow pointers, one pass: when fast reaches the end, slow has gone half as far.

In [ ]:
def middle_node(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    return slow


for values, expected in [([1, 2, 3, 4, 5], 3), ([1, 2, 3, 4, 5, 6], 4), ([1], 1), ([1, 2], 2)]:
    assert middle_node(from_list(values)).val == expected
for n in range(1, 30):
    assert middle_node(from_list(range(n))).val == n // 2          # matches the two-pass count
assert middle_node(None) is None
print("middle_node: odd, even, single and empty lists pass (second middle for even lengths)")

Changing the loop to `while fast.next and fast.next.next` returns the *first* middle instead — the version you want when splitting a list in half for merge sort.

## Problem 4 · Merge two sorted lists (LC 21)

**Statement:** merge two sorted lists by splicing their nodes. `1→2→4` + `1→3→4` → `1→1→2→3→4→4`.

**Optimal:** a **dummy head** and a `tail` pointer. Repeatedly attach the smaller front node; when one list runs out, attach the rest of the other in one step. Without the dummy, "which node is the head?" needs a special case.

In [ ]:
def merge_two_lists(l1, l2):
    dummy = tail = ListNode()
    while l1 and l2:
        if l1.val <= l2.val:                   # <= keeps equal values from l1 first: stable
            tail.next, l1 = l1, l1.next
        else:
            tail.next, l2 = l2, l2.next
        tail = tail.next
    tail.next = l1 or l2                       # attach whatever is left
    return dummy.next


for a, b in [([1, 2, 4], [1, 3, 4]), ([], []), ([], [0]), ([5], [1, 2, 3]), ([1, 1], [1])]:
    assert to_list(merge_two_lists(from_list(a), from_list(b))) == sorted(a + b)
for _ in range(300):
    a = sorted(rng.randint(0, 9) for _ in range(rng.randint(0, 8)))
    b = sorted(rng.randint(0, 9) for _ in range(rng.randint(0, 8)))
    assert to_list(merge_two_lists(from_list(a), from_list(b))) == sorted(a + b)
print("merge_two_lists: edge cases + 300 random pairs pass")

**Complexity:** O(m + n) time, O(1) extra space (nodes are re-linked, not copied). Merging k lists: a heap of the k front nodes, O(N log k) ([10 · Heaps](10_heaps_and_top_k.ipynb)).

## Problem 5 · Remove the n-th node from the end (LC 19)

**Statement:** remove the n-th node from the end, in one pass. `1→2→3→4→5`, n=2 → `1→2→3→5`.

**Brute force:** count the length L, then walk to node L − n — two passes. **Optimal:** two pointers with a **gap**: move `lead` n + 1 steps ahead of `trail` (both starting at a dummy), then move both until `lead` falls off the end — `trail` is now just *before* the node to delete. The dummy handles "remove the head" with no special case.

In [ ]:
def remove_nth_from_end(head, n):
    dummy = ListNode(0, head)
    lead = trail = dummy
    for _ in range(n + 1):
        lead = lead.next                       # a gap of n nodes between trail and lead
    while lead:
        lead, trail = lead.next, trail.next
    trail.next = trail.next.next               # unlink the target
    return dummy.next


for values, n, expected in [([1, 2, 3, 4, 5], 2, [1, 2, 3, 5]), ([1], 1, []), ([1, 2], 1, [1]), ([1, 2], 2, [2]), ([1, 2, 3], 3, [2, 3])]:
    assert to_list(remove_nth_from_end(from_list(values), n)) == expected
for _ in range(300):
    values = list(range(rng.randint(1, 12)))
    n = rng.randint(1, len(values))
    expected = values[:len(values) - n] + values[len(values) - n + 1:]
    assert to_list(remove_nth_from_end(from_list(values), n)) == expected
print("remove_nth_from_end: removing the head, the tail and the only node all pass")

## Problem 6 · Palindrome linked list (LC 234)

**Statement:** does the list read the same both ways? `1→2→2→1` → True; `1→2` → False. Aim for O(1) extra space.

**Brute force:** copy the values into a Python list and compare with its reverse — O(n) space. **Optimal:** find the middle (fast/slow), reverse the second half in place, compare the two halves, then **reverse it back** — a good citizen does not leave the caller's list mangled.

In [ ]:
def is_palindrome_list(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    second = reverse_iterative(slow)           # reverse from the middle to the end
    result, p, q = True, head, second
    while q:
        if p.val != q.val:
            result = False
            break
        p, q = p.next, q.next
    reverse_iterative(second)                  # restore the original list
    return result


for values in [[1, 2, 2, 1], [1, 2], [1], [], [1, 2, 3, 2, 1], [1, 2, 3, 1], [5, 5]]:
    head = from_list(values)
    assert is_palindrome_list(head) == (values == values[::-1]), values
    assert to_list(head) == values                                   # the input is left intact
for _ in range(300):
    values = [rng.randint(0, 2) for _ in range(rng.randint(0, 9))]
    head = from_list(values)
    assert is_palindrome_list(head) == (values == values[::-1]) and to_list(head) == values
print("is_palindrome_list: O(1) extra space, matches the copy-and-compare brute force, and restores the input")

## Problem 7 · Intersection of two lists (LC 160)

**Statement:** two lists may merge into a shared tail (a Y shape). Return the first shared **node** (by identity), or None.

```
 a:  4 -> 1 \
             8 -> 4 -> 5 -> None        the shared tail starts at the node holding 8
 b:  5 -> 6 -> 1 /
```

**Brute force:** put every node of `a` in a set; walk `b` until a node is in the set — O(m + n) time, O(m) space. **Optimal:** two pointers; when one reaches the end, it jumps to the *other* list's head. Both then walk `len(a) + len(b)` steps in total, so they arrive at the shared node at the same moment (or both reach None together if there is none). Lengths `x + c` and `y + c`: `x + c + y = y + c + x`.

In [ ]:
def intersection_node(a, b):
    p, q = a, b
    while p is not q:
        p = p.next if p else b
        q = q.next if q else a
    return p                                    # the shared node, or None


def intersection_brute(a, b):
    seen = set()
    while a:
        seen.add(id(a))
        a = a.next
    while b:
        if id(b) in seen:
            return b
        b = b.next
    return None


def make_y(prefix_a, prefix_b, shared):
    tail = from_list(shared)
    heads = []
    for prefix in (prefix_a, prefix_b):
        head = from_list(prefix)
        if head is None:
            heads.append(tail)
            continue
        node = head
        while node.next:
            node = node.next
        node.next = tail
        heads.append(head)
    return heads[0], heads[1], tail


a, b, shared = make_y([4, 1], [5, 6, 1], [8, 4, 5])
meet = intersection_node(a, b)
print("intersection at the node holding", meet.val, "| same object as the shared tail:", meet is shared)
assert meet is shared
assert intersection_node(from_list([1, 2]), from_list([1, 2])) is None     # equal VALUES, different nodes
for _ in range(300):
    shared_vals = list(range(rng.randint(0, 4)))
    a, b, _ = make_y(list(range(rng.randint(0, 5))), list(range(rng.randint(0, 5))), shared_vals)
    assert intersection_node(a, b) is intersection_brute(a, b)
print("intersection_node: O(1) space, matches the set-based brute force; equal values are not an intersection")

## When linked lists matter in Python (rarely)

A Python `list` is a dynamic array, not a linked list. Hand-written node classes are slow in CPython — every `node.next` is an attribute lookup on a separate heap object — so you almost never write one outside an interview. Where linked lists *do* show up:

| Where | What it is |
|---|---|
| `collections.deque` | a doubly linked list of 64-slot blocks: O(1) at both ends |
| `collections.OrderedDict` | a dict plus a doubly linked list of keys: O(1) `move_to_end` and `popitem(last=False)` |
| LRU caches (`functools.lru_cache`, LC 146) | hash map for O(1) lookup + doubly linked list for O(1) "most recently used" reordering |
| memory allocators, OS schedulers, undo histories | splicing nodes in O(1) when you already hold the node |

The trade-off, measured on 100,000 items: inserting at the front favours linked structures; walking all the values favours the contiguous list.

In [ ]:
def interleaved_times(calls, number=1, rounds=3):
    best = [float("inf")] * len(calls)
    for _ in range(rounds):
        for i, call in enumerate(calls):
            t0 = time.perf_counter()
            for _ in range(number):
                call()
            best[i] = min(best[i], (time.perf_counter() - t0) / number)
    return best


n = 100_000
py_list, dq, linked = list(range(n)), deque(range(n)), {"head": from_list(range(n))}


def linked_push_pop():
    linked["head"] = ListNode(0, linked["head"])        # insert at the front ...
    linked["head"] = linked["head"].next                # ... and remove it again


def walk_linked():
    total, node = 0, linked["head"]
    while node:
        total += node.val
        node = node.next
    return total


front = interleaved_times([lambda: (py_list.insert(0, 0), py_list.pop(0)), linked_push_pop,
                           lambda: (dq.appendleft(0), dq.popleft())], number=200)
walk = interleaved_times([lambda: sum(py_list), walk_linked])
assert walk_linked() == sum(py_list)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].bar(["list\ninsert(0)+pop(0)", "linked list\npush+pop front", "deque\nappendleft+popleft"],
            [t * 1e6 for t in front], color=COLORS[0], width=0.5)
axes[0].set(title="Insert + remove at the front (n = 100k)", ylabel="µs per operation")
axes[1].bar(["sum(list)", "walk the\nlinked list"], [t * 1e3 for t in walk], color=COLORS[1], width=0.4)
axes[1].set(title="Visit all 100k values", ylabel="ms")
for ax, values, unit in [(axes[0], [t * 1e6 for t in front], "µs"), (axes[1], [t * 1e3 for t in walk], "ms")]:
    ax.grid(axis="x", visible=False)
    for i, v in enumerate(values):
        ax.text(i, v, f"{v:.2f} {unit}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()
print(f"front insert: list {front[0] * 1e6:.1f} µs vs linked {front[1] * 1e6:.2f} µs | "
      f"full scan: list {walk[0] * 1e3:.2f} ms vs linked {walk[1] * 1e3:.2f} ms")
assert front[1] < front[0] and front[2] < front[0]     # O(1) beats O(n) at the front
assert walk[0] < walk[1]                               # contiguous memory + C loop beats pointer chasing

The LRU cache is the one design question to be ready for. With `OrderedDict` it is ten lines; [14 · Tries, union-find and LRU cache](14_tries_union_find_lru_cache.ipynb) builds the hash map + doubly linked list version from scratch.

In [ ]:
class LRUCache:
    """O(1) get/put: OrderedDict keeps keys in usage order (a doubly linked list inside)."""

    def __init__(self, capacity):
        self.capacity = capacity
        self.data = OrderedDict()

    def get(self, key):
        if key not in self.data:
            return -1
        self.data.move_to_end(key)                 # mark as most recently used
        return self.data[key]

    def put(self, key, value):
        self.data[key] = value
        self.data.move_to_end(key)
        if len(self.data) > self.capacity:
            self.data.popitem(last=False)          # evict the least recently used


cache = LRUCache(2)
cache.put(1, 1)
cache.put(2, 2)
results = [cache.get(1)]
cache.put(3, 3)                                    # evicts key 2 (least recently used)
results.append(cache.get(2))
cache.put(4, 4)                                    # evicts key 1
results += [cache.get(1), cache.get(3), cache.get(4)]
print("LeetCode 146 sequence ->", results)
assert results == [1, -1, -1, 3, 4]

## Pattern cheat sheet — if you see X, think Y

| If you see... | Think... |
|---|---|
| the head might change (delete, merge, insert in front) | dummy node, return `dummy.next` |
| middle of the list, split in half | fast & slow pointers |
| cycle? where does it start? | Floyd: meet, then reset one pointer to the head |
| n-th / k-th from the end in one pass | two pointers with a gap of n |
| reverse all or a section | `prev, cur, nxt` — save next *before* rewiring |
| palindrome / reorder / "fold" the list | middle + reverse the second half (+ restore) |
| two lists share a tail | switch heads at the end (or align by length difference) |
| merge sorted lists | dummy + tail; k lists → heap |
| O(1) lookup **and** O(1) reordering | hash map + doubly linked list (LRU) — `OrderedDict` in Python |
| "same node?" | compare with `is`, not `==` on values |

## Try it yourself

**1. Remove linked-list elements (LC 203).** Remove every node whose value equals `val`: `1→2→6→3→4→5→6`, val 6 → `1→2→3→4→5`. (The head may be removed too — use a dummy.)

In [ ]:
# Solution — try it yourself first, then run this
def remove_elements(head, val):
    dummy = prev = ListNode(0, head)
    while prev.next:
        if prev.next.val == val:
            prev.next = prev.next.next        # unlink; stay put to check the new next
        else:
            prev = prev.next
    return dummy.next


for values, val in [([1, 2, 6, 3, 4, 5, 6], 6), ([], 1), ([7, 7, 7, 7], 7), ([1, 7, 7, 2], 7), ([1, 2], 3)]:
    assert to_list(remove_elements(from_list(values), val)) == [v for v in values if v != val]
print("remove_elements: all cases pass (including removing every node)")

**2. Add two numbers (LC 2).** Digits are stored in reverse order, one per node. Return the sum as a list: `2→4→3` (342) + `5→6→4` (465) → `7→0→8` (807).

In [ ]:
# Solution — try it yourself first, then run this
def add_two_numbers(l1, l2):
    dummy = tail = ListNode()
    carry = 0
    while l1 or l2 or carry:
        total = carry + (l1.val if l1 else 0) + (l2.val if l2 else 0)
        carry, digit = divmod(total, 10)
        tail.next = ListNode(digit)
        tail = tail.next
        l1 = l1.next if l1 else None
        l2 = l2.next if l2 else None
    return dummy.next


def as_digits(n):
    return [int(d) for d in reversed(str(n))]


assert to_list(add_two_numbers(from_list([2, 4, 3]), from_list([5, 6, 4]))) == [7, 0, 8]
assert to_list(add_two_numbers(from_list([9, 9, 9]), from_list([1]))) == [0, 0, 0, 1]     # the final carry
for _ in range(300):
    x, y = rng.randint(0, 10 ** 6), rng.randint(0, 10 ** 6)
    assert to_list(add_two_numbers(from_list(as_digits(x)), from_list(as_digits(y)))) == as_digits(x + y)
print("add_two_numbers: all cases pass, including a carry into a new digit")

**3. Reorder list (LC 143).** Rearrange `L0→L1→…→Ln` into `L0→Ln→L1→Ln−1→…` in place: `1→2→3→4→5` → `1→5→2→4→3`. It combines three tricks: middle, reverse, merge.

In [ ]:
# Solution — try it yourself first, then run this
def reorder_list(head):
    if not head or not head.next:
        return head
    slow, fast = head, head
    while fast.next and fast.next.next:        # first middle
        slow, fast = slow.next, fast.next.next
    second = reverse_iterative(slow.next)
    slow.next = None                           # cut the list in two
    first = head
    while second:                              # weave the halves together
        first_next, second_next = first.next, second.next
        first.next, second.next = second, first_next
        first, second = first_next, second_next
    return head


def reorder_reference(values):
    out, lo, hi = [], 0, len(values) - 1
    while lo <= hi:
        out.append(values[lo])
        if lo != hi:
            out.append(values[hi])
        lo, hi = lo + 1, hi - 1
    return out


for values in [[1, 2, 3, 4], [1, 2, 3, 4, 5], [1], [], [1, 2]]:
    assert to_list(reorder_list(from_list(values))) == reorder_reference(values), values
print("reorder_list:", to_list(reorder_list(from_list([1, 2, 3, 4, 5]))))

## Say it in an interview

- **30-second answer:** "A linked list trades O(1) indexing for O(1) insert and delete at a node you already hold. The interview problems reduce to three tricks: a dummy head so the first node is not a special case, fast and slow pointers for middles, cycles and 'n-th from the end', and in-place reversal with prev/cur/next."
- **Floyd in one breath:** "slow moves one, fast moves two; inside a loop fast gains one step per move, so they meet. Then a + b is a whole number of laps, so a pointer from the head and one from the meeting point, one step at a time, meet at the cycle start. O(n) time, O(1) space."
- **Key facts:** reverse is O(n) time, O(1) space iteratively, O(n) stack recursively; merge is O(m + n) with O(1) extra; the palindrome check is O(1) extra if you reverse half (and restore it).
- **Follow-ups:** "Doubly linked?" — O(1) delete given the node (no need to find its predecessor), which is why LRU caches use one. "In Python?" — you would use `deque` or `OrderedDict`; hand-rolled nodes lose to contiguous lists on scans because of pointer chasing.
- **Traps:** losing the rest of the list by overwriting `next` before saving it; comparing nodes by value instead of identity; forgetting `fast.next` in the loop condition; not handling the empty list and the one-node list.

## Next

- [06 · Binary search](06_binary_search.ipynb)
- [09 · Trees and BST](09_trees_and_bst.ipynb) — nodes with two pointers instead of one
- [14 · Tries, union-find and LRU cache](14_tries_union_find_lru_cache.ipynb) — the LRU cache built from a hash map and a doubly linked list
- [04 · Stacks and queues](04_stacks_and_queues.ipynb) — `deque`, the linked structure you actually use in Python
- Theory: [DSA basics course](../../dsa_basics/index.html) (chapter 4, "Nodes & linked lists")